In [22]:
from models import ( 
    KNNClassifier, 
    DecisionTree, 
    RandomForest, 
    ArtificialNeuralNetwork, 
    LogisticRegression
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from matplotlib import pyplot as plt

import numpy as np
import pandas as pd
import joblib

X_train = pd.read_csv(r"data\cleaned\X_train_scaled.csv", index_col=0)
X_val = pd.read_csv(r"data\cleaned\X_val_scaled.csv", index_col=0)
y_train = pd.read_csv(r"data\cleaned\y_train.csv", index_col=0)
y_val = pd.read_csv(r"data\cleaned\y_val.csv", index_col=0)

base_models = {
    "decision-tree": joblib.load("output/dtl1.pkl"),
    "random-forest": joblib.load("output/rf1.pkl"),
    "knn": joblib.load("output/knn_model_diy.pkl"),
    "logistic-regression": joblib.load("output/logistic_model_diy.pkl"),
    "ann": joblib.load("output/ann_relu_model.pkl")
}

knn_config = joblib.load("output/knn_model_diy_config.pkl")
knn_metric = knn_config["metric"]

In [23]:
print("Konfigurasi KNN:", knn_config)

Konfigurasi KNN: {'metric': 'manhattan', 'k': 15}


In [24]:
def tree_features(node):
    if not isinstance(node, dict):
        return set()

    return (
        {node["feature"]}
        | tree_features(node["children"]["left"])
        | tree_features(node["children"]["right"])
    )


used_features = tree_features(base_models["decision-tree"].tree)

for tree in base_models["random-forest"].forest:
    used_features |= tree_features(tree.tree)

unexpected = used_features - set(X_train.columns)

print("Fitur pohon di luar fitur training:", unexpected)

Fitur pohon di luar fitur training: {'Curricular units 1st sem (credited)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (without evaluations)', 'Curricular units 1st sem (without evaluations)'}


In [25]:
base_models["decision-tree"] = DecisionTree(max_depth=1)
base_models["decision-tree"].fit(X_train, y_train)

base_models["random-forest"] = RandomForest()
base_models["random-forest"].fit(X_train, y_train)

Tree 10 of 10...

In [26]:
def predict_model(X):
    return pd.DataFrame(
        {
            "decision-tree" : base_models["decision-tree"].predict(X),
            "random-forest" : base_models["random-forest"].predict(X),
            "knn" : base_models["knn"].predict(X, distance_metric=knn_metric, batch_size=64),
            "logistic-regression" : base_models["logistic-regression"].predict(X),
            "ann" : base_models["ann"].predict(X.to_numpy(dtype=float))
        }, 
        index=X.index
    )

val_predictions = predict_model(X_val)

val_predictions.head()

,decision-tree,random-forest,knn,logistic-regression,ann
1065,0,0,0,0,0
2990,2,2,2,2,2
2808,2,2,2,2,2
172,2,2,2,2,1
2712,2,2,2,2,0


## **Normal Vote**

In [27]:
def vote(predictions):
    return (predictions.mode(axis=1).iloc[:, 0].to_numpy(dtype=int))

std_vote_pred = vote(val_predictions)

In [28]:
results = []

for name in val_predictions.columns:
    pred = val_predictions[name].to_numpy()

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "macro_f1": f1_score(
            y_val, pred, average="macro", zero_division=0
        ),
    })

results.append({
    "model": "ensemble",
    "accuracy": accuracy_score(y_val, std_vote_pred),
    "macro_f1": f1_score(
        y_val,
        std_vote_pred,
        average="macro",
        zero_division=0,
    ),
})

comparison = pd.DataFrame(results).sort_values(
    "macro_f1", ascending=False
)

display(comparison)

print(classification_report(
    y_val,
    std_vote_pred,
    labels=[0, 1, 2],
    target_names=["Dropout", "Enrolled", "Graduate"],
    zero_division=0,
))

,model,accuracy,macro_f1
3,logistic-regression,0.753230,0.655300
2,knn,0.741602,0.636303
5,ensemble,0.729974,0.571111
4,ann,0.631783,0.551358
1,random-forest,0.711886,0.514684
0,decision-tree,0.704134,0.507326


              precision    recall  f1-score   support

     Dropout       0.72      0.78      0.75       249
    Enrolled       0.61      0.08      0.14       139
    Graduate       0.74      0.94      0.83       386

    accuracy                           0.73       774
   macro avg       0.69      0.60      0.57       774
weighted avg       0.71      0.73      0.68       774



## **Weighted Vote**

In [29]:
from sklearn.metrics import accuracy_score

accuracies = pd.Series({
    name: accuracy_score(y_val, val_predictions[name]) for name in val_predictions.columns
})

weights = accuracies / accuracies.sum()

display(pd.DataFrame({
    "validation_accuracy": accuracies,
    "weight": weights,
}).sort_values("weight", ascending=False))

,validation_accuracy,weight
logistic-regression,0.753230,0.212619
knn,0.741602,0.209336
random-forest,0.711886,0.200948
decision-tree,0.704134,0.198760
ann,0.631783,0.178337


In [30]:
def weighted_vote(predictions, weights, classes):
    weights = weights.loc[predictions.columns].to_numpy(dtype=float)
    classes = np.asarray(classes)
    pred_array = predictions.to_numpy()

    scores = np.column_stack([ (pred_array == label) @ weights for label in classes])

    winners = classes[np.argmax(scores, axis=1)]
    
    return winners, pd.DataFrame(
        scores,
        index=predictions.index,
        columns=classes,
    )

In [31]:
classes = np.sort(y_train["Target"].unique())
weight_pred, weight_score = weighted_vote(val_predictions, weights, classes) 

## **Perbandingan Hasil Voting**

In [32]:
comparison = pd.DataFrame([
    {
        "method": "Hard voting",
        "accuracy": accuracy_score(y_val, std_vote_pred),
        "macro_f1": f1_score(
            y_val, std_vote_pred, average="macro", zero_division=0
        ),
    },
    {
        "method": "Weighted hard voting",
        "accuracy": accuracy_score(y_val, weight_pred),
        "macro_f1": f1_score(
            y_val, weight_pred, average="macro", zero_division=0
        ),
    }
])

display(comparison)

,method,accuracy,macro_f1
0,Hard voting,0.729974,0.571111
1,Weighted hard voting,0.731266,0.579388


In [41]:
test_df = pd.read_csv("data/cleaned/test_df.csv", index_col=0)
X_test = test_df.loc[:, X_train.columns]

test_predictions = predict_model(X_test)
weight_pred, test_scores = weighted_vote(test_predictions,weights,classes)
weight_pred = np.asarray(weight_pred)

target_labels = pd.Series(weight_pred).map({
    0: "Dropout",
    1: "Enrolled",
    2: "Graduate",
})

submission = pd.DataFrame({
    "Student_ID": test_df["Student_ID"].to_numpy(),
    "Target": target_labels.to_numpy(),
})

submission.head()

,Student_ID,Target
0,2701,Dropout
1,1142,Graduate
2,3538,Dropout
3,3000,Dropout
4,476,Graduate


In [42]:
submission.to_csv("output/csv/submission_ensemble_weighted.csv", index=False, )